# **Abiotic Data Analysis**
Here, we pull salinity, sea surface temperature (SST), and precipitation data collected throughout the study and measure its impact on anemone algal density and chlorophyll concentration

## The notebook is broken into three steps:
### **Step 1) Pull and Process Abiotic Data** 
For each of the following data variables, we calculate daily and weekly averages:
* On-site SST from HOBO logger
* On-site SST and salinity from Star Oddi logger
* Off-site salinity from Fort Point buoy (located in SF Bay)
* Near-site precipitation from nearby terrestrial datasets

We also pull SST data from one of NOAA’s ERDDAP satellite datasets (Dataset ID: jplMURSST41). We also calulate daily and weekly mean on this data.

### **Step 2) Visualize collected SST and Salinity data**
### **Step 3a) Regression Analysis - Setup**
* Pull, calculate average, and merge algal density and chlorophyll concentration data together
* Merge with weekly mean Star Oddi SST and Salinity Data
    * infill missing SST values with Satellite SST
### **Step 3b) Regression Analysis - Calculation & Visaulization**
Calculate and plot regressions for the following:
* Salinity on Algal Density
* Temperature on Algal Density
* Salinity on Chlorophyll Concentration
* Temperature on Chlorophyll Concentration

In [ ]:
# load some library
import sys
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import xarray as xr

sys.path.append(os.path.expanduser('../../'))
from utils.functions import process_abiotic_data, site_sample_area
from utils.visuals import abiotic_plot, regression_plot

## **Step 1) Pull and Process Abiotic Data**
* Set collection start and end dates (with a few additional weeks buffer)
* Calculate weekly and daily averages for each dataset

In [ ]:
abiotic_data_path = 'abiotic_data/'
start_date = '2022-08-01T00:00:00'
end_date = '2023-03-31T00:00:00'
data, seven_day_averages_data, daily_averages_data = process_abiotic_data(abiotic_data_path, start_date, end_date)

In [ ]:
# Accessing the datasets
hobo_data = data['hobo']
star_oddi_data = data['star_oddi']
fort_point_daily_data = data['fort_point_daily']
fort_point_hourly_data = data['fort_point_hourly']
precipitation_data = data['precipitation']

hobo_seven_day_average_data = seven_day_averages_data['hobo_seven_day_average']
star_oddi_seven_day_average_data = seven_day_averages_data['star_oddi_seven_day_average']
fort_point_seven_day_average_data = seven_day_averages_data['fort_point_daily_seven_day_average']
fort_point_seven_day_hourly_average_data = seven_day_averages_data['fort_point_hourly_seven_day_average']
precipitation_seven_day_average_data = seven_day_averages_data['precipitation_seven_day_average']

hobo_average_data = daily_averages_data['hobo_daily_average']
star_oddi_average_data = daily_averages_data['star_oddi_daily_average']
fort_point_daily_average_data = daily_averages_data['fort_point_daily_daily_average']
fort_point_hourly_average_data = daily_averages_data['fort_point_hourly_daily_average']
precipitation_average_data = daily_averages_data['precipitation_daily_average']

## **Pull SST data from one of NOAA’s ERDDAP satellite datasets**

## Open a pointer to open the dataset

* These new variables will hold all the data for that link
* We'll run one below so you can see what it looks like
---

In [ ]:
mur_sst_url = 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/jplMURSST41'
mur_anom_month_url = 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/jplMURSST41anommday'
mur_sst_month__url = 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/jplMURSST41mday'

sst = xr.open_dataset(mur_sst_url)
month_sst_anom = xr.open_dataset(mur_anom_month_url)
month_sst = xr.open_dataset(mur_sst_month__url)

### Create a small square area for satellite data to retrieve from, offshore about 5km

In [ ]:
start_time = '2022-08-27T12:00:00'
end_time = '2023-04-22T12:00:00'
data = sst

ap_beach = site_sample_area(data=data, start_time=start_time, end_time=end_time,site='ap')

### Calculate Daily and Weekly SST Averages

In [ ]:
# Average over lat/lon to get a time series of daily SST
daily_satellite_sst = ap_beach.mean(dim=['latitude', 'longitude']).to_series()
daily_satellite_sst.index = pd.to_datetime(daily_satellite_sst.index)  # Ensure datetime index

satellite_sst_7day_avg = daily_satellite_sst.rolling(window=7, min_periods=1).mean()

# Step 1: Turn your SST series into a DataFrame
satellite_sst_df = satellite_sst_7day_avg.reset_index()
satellite_sst_df.columns = ['date_time', 'satellite_temp_avg']

## **Step 2) Visualize collected SST and Salinity data**

### Visualize SST from data loggers and satellite data

In [ ]:
data_dict = {
    'Star Oddi Temperature Logger': (star_oddi_average_data['date_time'], star_oddi_average_data['temp_c_daily_average'], 'royalblue', 'line'),
    'HOBO Temperature Logger': (hobo_average_data['date_time'], hobo_average_data['temp_c_daily_average'], 'orange', 'line'),
    'Offshore Satellite Data': (ap_beach.time, ap_beach.mean(axis=(1,2)), 'indigo', 'line')
}

abiotic_plot('line', data_dict, 
            #'Fall Temperature Decrease Similarly Detected \n with Satellite and Intertidal Loggers', 
            xlabel='Date', ylabel='Temp (°C)', 
            xlim=(start_date, end_date), 
            ylim=(8, 20), 
            save_path='plots/temperature_trends_no_title.png')

### Visualize Salinity from data loggers with Local Terrestrial Rainfall Data

In [ ]:
data_dict = {
    'Average Daily Intertidal Salinity': (star_oddi_average_data['date_time'], star_oddi_average_data['salinity_ppt_daily_average'], 'royalblue', 'scatter'),
    'Fort Point Buoy Salinity': (fort_point_daily_average_data['date_time'], fort_point_daily_average_data['salinity_ppt_daily_average'], 'red', 'scatter'),
    'rain': (precipitation_data['date_time'], precipitation_data['rain_mm'], 'orange', 'bar')
}

abiotic_plot('scatter', data_dict, 
            #'Local Rain Events Drastically Drop Intertidal Logger Salinity', 
            xlabel='Date', ylabel='Salinity (psu)', 
            xlim=(start_date, end_date), 
            save_path='figures/salinity_rainfall_no_title.png')

## **Step 3a) Regression Analysis - Setup**

Pull cleaned algal density and chlorophyll concentration datasets from `biotic_data_analysis.ipynb`

In [ ]:
a_sola_algae_data = pd.read_csv('biotic_data/a_sola_algae_data.csv')
a_sola_algae_data = a_sola_algae_data[['date_time', 'num_cells_per_ug_protein']]

a_sola_chl_data = pd.read_csv('biotic_data/a_sola_chlorophyll_data.csv')
a_sola_chl_data = a_sola_chl_data[['date_time', 'ng_chlorophyll_per_ug_protein']]

### Calculate mean and merge both `algal density` and `chlorophyll` datasets together

In [ ]:
# Ensure date_time is treated as a datetime object
a_sola_chl_data['date_time'] = pd.to_datetime(a_sola_chl_data['date_time'])
a_sola_algae_data['date_time'] = pd.to_datetime(a_sola_algae_data['date_time'])

# Group by date_time and calculate the mean
average_chl_by_date = a_sola_chl_data.groupby('date_time', as_index=False)['ng_chlorophyll_per_ug_protein'].mean()
average_algae_by_date = a_sola_algae_data.groupby('date_time', as_index=False)['num_cells_per_ug_protein'].mean()

average_algae_by_date = average_algae_by_date.rename(columns={'num_cells_per_ug_protein':'avg_num_cells_per_ug_protein'})
average_chl_by_date = average_chl_by_date.rename(columns={'ng_chlorophyll_per_ug_protein':'avg_ng_chlorophyll_per_ug_protein'})

avg_biotic_dependent_vars = pd.merge(average_algae_by_date, average_chl_by_date, on='date_time', how='outer')
avg_biotic_dependent_vars.head()

### Merge our biotic dataset with weekly averaged Star Oddi SST and salinity data

Given the closely associated temperature values across our satellite, Star Oddi, and HOBO datasets, we use Star Oddi for our regression analysis as it also has our on-site salinity data 

**Note:** We infill missing (NaN) sst values with satellite data as it is our only continuous SST dataset

In [ ]:
# Sort both DataFrames by date_time (required for merge_asof)
star_oddi_weekly = star_oddi_seven_day_average_data.sort_values('date_time')

# Perform an asof merge to find the closest matching date
avg_abiotic_biotic_data = pd.merge_asof(avg_biotic_dependent_vars, star_oddi_weekly, on='date_time', direction='nearest')

In [ ]:
# Standardize both to nanoseconds
avg_abiotic_biotic_data['date_time'] = avg_abiotic_biotic_data['date_time'].astype('datetime64[ns]')
satellite_sst_df['date_time'] = satellite_sst_df['date_time'].astype('datetime64[ns]')


# Ensure both are sorted by date
avg_abiotic_biotic_data = avg_abiotic_biotic_data.sort_values('date_time')
satellite_sst_df = satellite_sst_df.sort_values('date_time')

# Use merge_asof to find nearest SST for each algae sample date
avg_abiotic_biotic_data = pd.merge_asof(
    avg_abiotic_biotic_data,
    satellite_sst_df,
    on='date_time',
    direction='nearest',       
    tolerance=pd.Timedelta('7D')  # Only use SST if it's within 7 days
)

# Fill NaNs with the satellite SST values
avg_abiotic_biotic_data['temp_c_seven_day_average'] = avg_abiotic_biotic_data['temp_c_seven_day_average'].fillna(
    avg_abiotic_biotic_data['satellite_temp_avg']
)

# Drop the satellite column afterward
avg_abiotic_biotic_data = avg_abiotic_biotic_data.drop(columns=['satellite_temp_avg'])

## **Step 3b) Regression Analysis - Calculation & Visualization**

### Calculate and Visaulize Regression(s) - **Temperature and Salinity** on **Algal Density**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regression_plot(avg_abiotic_biotic_data, 
                   'salinity_ppt_seven_day_average', 
                   'avg_num_cells_per_ug_protein',
                   color='#4eb3d3', 
                   title='Algal Cell Density vs. Salinity \n (7-day average prior to sampling)',
                   ax=axes[0]
)                

regression_plot(avg_abiotic_biotic_data, 
                   'temp_c_seven_day_average', 
                   'avg_num_cells_per_ug_protein',
                   color='#4eb3d3', 
                   title='Algal Cell Density vs. Temperature \n (7-day average prior to sampling)',
                   ax=axes[1])


plt.tight_layout()
fig.savefig("plots/subplot_algae_salinity_temp_regression.png", dpi=300, bbox_inches="tight")
plt.show()

### Calculate and Visaulize Regression(s) - **Temperature and Salinity** on **Chlorophyll Concentration**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regression_plot(avg_abiotic_biotic_data, 
                   'salinity_ppt_seven_day_average', 
                   'avg_ng_chlorophyll_per_ug_protein',
                   color='#2c7fb8', 
                   title='Chlorophyll α Concentration vs. Salinity \n (7-day average prior to sampling)',
                   ax=axes[0])

regression_plot(avg_abiotic_biotic_data, 
                   'temp_c_seven_day_average', 
                   'avg_ng_chlorophyll_per_ug_protein',
                   color='#2c7fb8', 
                   title='Chlorophyll α Concentration vs. Temperature \n (7-day average prior to sampling)',
                   ax=axes[1])

plt.tight_layout()
fig.savefig("plots/subplot_chl_salinity_temp_regression.png", dpi=300, bbox_inches="tight")
plt.show()